# KG1 Nemotron v39 GRPO Training

**Objetivo:** GRPO sobre v30 LoRA (0.68 LB) para 0.75+

**Requisitos:**
1. Runtime > Change runtime type > **A100 GPU** (40GB minimo!)
2. Secrets: `HF_KEY`, `KAGGLE_KEY`, `KAGGLE_USERNAME`
3. Run All

**Config:**
- 4-bit NF4 quantization (~15GB VRAM)
- v30 LoRA continuado (sem merge - fix bug v38)
- beta=0.0 | lr=5e-6 | 300 steps
- Binary + format reward
- Auto-submit Kaggle a cada 50 steps
- Smart data filtering (foco cipher/equation)

In [ ]:
#@title 1. Instalar dependencias
import subprocess, sys, os

# Fix pyarrow BEFORE anything imports it
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "pyarrow"],
               capture_output=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyarrow"],
               capture_output=True)

import torch
print(f"PyTorch: {torch.__version__}, CUDA: {torch.version.cuda}")

# Auto-detect Python/Torch/ABI for mamba-ssm wheels
py_ver = f"cp{sys.version_info.major}{sys.version_info.minor}"
torch_ver = ".".join(torch.__version__.split(".")[:2])
cxx11 = "TRUE" if torch._C._GLIBCXX_USE_CXX11_ABI else "FALSE"
print(f"Python: {py_ver}, Torch: {torch_ver}, CXX11_ABI: {cxx11}")

CAUSAL_CONV1D_WHEEL = (
    f"https://github.com/Dao-AILab/causal-conv1d/releases/download/v1.6.1.post4/"
    f"causal_conv1d-1.6.1%2Bcu12torch{torch_ver}cxx11abi{cxx11}-"
    f"{py_ver}-{py_ver}-linux_x86_64.whl"
)
MAMBA_SSM_WHEEL = (
    f"https://github.com/state-spaces/mamba/releases/download/v2.3.1/"
    f"mamba_ssm-2.3.1%2Bcu12torch{torch_ver}cxx11abi{cxx11}-"
    f"{py_ver}-{py_ver}-linux_x86_64.whl"
)

for name, url in [("causal-conv1d", CAUSAL_CONV1D_WHEEL),
                   ("mamba-ssm", MAMBA_SSM_WHEEL)]:
    print(f"  {name}: {url.split('/')[-1]}")
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", url],
                       capture_output=True, text=True)
    if r.returncode == 0:
        print(f"    OK")
    else:
        alt_abi = "FALSE" if cxx11 == "TRUE" else "TRUE"
        alt_url = url.replace(f"cxx11abi{cxx11}", f"cxx11abi{alt_abi}")
        r2 = subprocess.run([sys.executable, "-m", "pip", "install", "-q", alt_url],
                            capture_output=True, text=True)
        if r2.returncode == 0:
            print(f"    OK (alt ABI)")
        else:
            print(f"    Building from source (~5min)...")
            subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                          "ninja", "packaging"], capture_output=True)
            r3 = subprocess.run([sys.executable, "-m", "pip", "install", "-q", name],
                                capture_output=True, text=True)
            if r3.returncode == 0:
                print(f"    OK (source)")
            else:
                raise RuntimeError(f"{name} FAILED: {r3.stderr[:200]}")

import mamba_ssm
print(f"mamba_ssm: {mamba_ssm.__version__}")

subprocess.run([sys.executable, "-m", "pip", "install", "-q",
    "transformers>=4.48.0", "peft>=0.14.0", "datasets", "accelerate>=1.2.0",
    "trl>=0.29.0", "huggingface_hub", "safetensors", "sentencepiece", "protobuf",
    "kaggle", "pandas", "bitsandbytes"],
    check=True, stdout=subprocess.DEVNULL)

# Verify datasets+pyarrow
try:
    from datasets import Dataset
    Dataset.from_dict({"a": [1, 2]})
    print("datasets+pyarrow: OK")
except Exception:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                   "--force-reinstall", "pyarrow==17.0.0"], capture_output=True)
    from datasets import Dataset
    Dataset.from_dict({"a": [1, 2]})
    print("pyarrow: FIXED (17.0.0)")

# Version check
import transformers, peft, trl, accelerate, bitsandbytes
print(f"\ntransformers={transformers.__version__} peft={peft.__version__} "
      f"trl={trl.__version__} accelerate={accelerate.__version__} "
      f"bnb={bitsandbytes.__version__}")
print("Dependencias OK!")

In [ ]:
#@title 2. GPU Check + Auth + Config
import torch, json, random, time, zipfile, re, os, sys, subprocess
import pandas as pd
from huggingface_hub import HfApi, login, hf_hub_download

# === GPU CHECK ===
if not torch.cuda.is_available():
    raise RuntimeError("CUDA indisponivel! Selecione GPU no Runtime.")

gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU: {gpu_name} ({vram_gb:.1f} GB)")

if vram_gb < 30:
    raise RuntimeError(
        f"\n{'='*55}\n"
        f"  ERRO: {gpu_name} tem apenas {vram_gb:.0f}GB VRAM\n"
        f"  MINIMO NECESSARIO: A100 (40GB) ou H100 (80GB)\n\n"
        f"  >> Runtime > Change runtime type > A100 GPU\n"
        f"  >> Marcar High-RAM se disponivel\n"
        f"  >> Disconnect and delete runtime > Run All\n"
        f"{'='*55}"
    )
print("GPU: OK!")

# === AUTH ===
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_KEY")
    KAGGLE_KEY = userdata.get("KAGGLE_KEY")
    KAGGLE_USERNAME = userdata.get("KAGGLE_USERNAME")
    print("Secrets: Colab")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN", os.environ.get("HF_KEY", ""))
    KAGGLE_KEY = os.environ.get("KAGGLE_KEY", "")
    KAGGLE_USERNAME = os.environ.get("KAGGLE_USERNAME", "felipe1983")

if HF_TOKEN:
    login(token=HF_TOKEN)
    print("HF: OK")
else:
    print("AVISO: Sem HF token - uploads vao falhar")

if KAGGLE_KEY:
    os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
    kpath = os.path.expanduser("~/.kaggle/kaggle.json")
    with open(kpath, "w") as f:
        json.dump({"username": KAGGLE_USERNAME, "key": KAGGLE_KEY}, f)
    os.chmod(kpath, 0o600)
    print("Kaggle: OK")
else:
    print("AVISO: Sem Kaggle key - submits vao falhar")

# Flash attention shim (evita ImportError)
try:
    from transformers.utils.import_utils import is_flash_attn_greater_or_equal_2_10
except ImportError:
    import transformers.utils.import_utils as _tiu
    _tiu.is_flash_attn_greater_or_equal_2_10 = lambda: False

api = HfApi()

# === CONFIG ===
DATA_REPO = "felipesp1983/kg1-nemotron-training"
SFT_REPO = "felipesp1983/kg1-nemotron-lora-v30-perfected"
SFT_CKPT = "checkpoint-400"
OUTPUT_REPO = "felipesp1983/kg1-nemotron-lora-v39-grpo"
MODEL_NAME = "nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16"
COMPETITION = "nvidia-nemotron-model-reasoning-challenge"
GRPO_STEPS = 150                                       # SPEED FIX: 300->150
SUBMIT_STEPS = [30, 60, 90, 120, 150]                  # SPEED FIX: checkpoints mais frequentes

# FIX CRITICO: Usar EXATAMENTE o mesmo prompt suffix que a avaliacao Kaggle!
# Fonte: src/competition_utils.py linha 18 (OFFICIAL_PROMPT_SUFFIX)
BOXED_SUFFIX = (
    "\nPlease put your final answer inside `\\boxed{}`. "
    "For example: `\\boxed{your answer}`"
)

print(f"\nConfig OK | {GRPO_STEPS} steps | GPU: {gpu_name}")
print(f"BOXED_SUFFIX: {BOXED_SUFFIX[:50]}...")


In [ ]:
#@title 3. Carregar dados + curriculum GRPO otimizado
from datasets import Dataset

print("=== Loading data ===")
hf_hub_download(repo_id=DATA_REPO, repo_type="dataset",
                filename="data/train.csv", local_dir="/tmp/kg1_data")
train_df = pd.read_csv("/tmp/kg1_data/data/train.csv")
print(f"Total: {len(train_df)} problems")

def classify(prompt):
    p = str(prompt).lower()
    if "bit manipulation" in p: return "bit"
    if "gravitational" in p or "gravity" in p: return "grav"
    if "unit conversion" in p or "measurement" in p: return "unit"
    if "numeral" in p: return "num"
    if "encryption" in p or "cipher" in p: return "enc"
    if "transformation" in p: return "eq"
    return "other"

train_df["family"] = train_df["prompt"].apply(classify)
train_df["ans_len"] = train_df["answer"].astype(str).str.len()

# FIX: ans_len <= 40 (max no dataset = 39, cobre 100%)
# O filtro antigo (<=24) eliminava 57% dos problemas cipher!
grpo_df = train_df[train_df["ans_len"] <= 40].copy()
print(f"All answers (max len {train_df['ans_len'].max()}): {len(grpo_df)} kept")

# === SMART FILTERING PARA GRPO ===
# CRITICO: Cipher ja esta em 100% accuracy - NÃO incluir no GRPO!
# (advantage = reward - baseline = 1.0 - 1.0 = 0 -> zero learning)
# Foco: equation (5-17% baseline, MAIOR gap) + bit (10-80% baseline)
# Easy (num/unit/grav): minimo para regularizacao

# equation: TODOS - maior gap absoluto, GRPO tem 53% chance de sinal com n=4
eq_df = grpo_df[grpo_df["family"] == "eq"]

# bit: 300 amostras - sinal fraco mas importante
bit_full = grpo_df[grpo_df["family"] == "bit"]
bit_df = bit_full.sample(n=min(300, len(bit_full)), random_state=42)

# easy (num/unit/grav): 100 amostras para regularizacao apenas
easy_full = grpo_df[grpo_df["family"].isin({"num", "unit", "grav"})]
easy_df = easy_full.sample(n=min(100, len(easy_full)), random_state=42)

# NÃO incluir cipher (enc) - ja 100% accuracy, zero gradient!
# NÃO incluir other
grpo_df = pd.concat([eq_df, bit_df, easy_df])
grpo_df = grpo_df.sample(frac=1, random_state=42)
print(f"GRPO dataset: {len(grpo_df)} problems (SEM cipher!)")

# Build HF dataset
prompts, answers = [], []
for _, row in grpo_df.iterrows():
    prompts.append([{"role": "user",
                     "content": str(row["prompt"]) + BOXED_SUFFIX}])
    answers.append(str(row["answer"]).strip())

grpo_dataset = Dataset.from_dict({"prompt": prompts, "answer": answers})

print("\nDistribuicao GRPO:")
for fam, cnt in grpo_df["family"].value_counts().items():
    pct = cnt / len(grpo_df) * 100
    print(f"  {fam:6s}: {cnt:5d} ({pct:.1f}%)")
print(f"\n  enc/cipher: EXCLUIDO (ja 100% accuracy)")
print(f"  Foco: eq ({len(eq_df)}) + bit ({len(bit_df)}) + easy ({len(easy_df)})")

In [ ]:
#@title 4. Reward functions
def extract_boxed(text):
    """Extract last \\boxed{...} from text."""
    matches = re.findall(
        r'\\boxed\{([^{}]*(?:\{[^{}]*\}[^{}]*)*)\}', text)
    return matches[-1].strip() if matches else None

def normalize_answer(ans):
    if ans is None:
        return None
    return str(ans).strip().strip('"\'')

def verify_answer(predicted, ground_truth):
    """Exact match for strings/binary. 1% tolerance for floats."""
    if predicted is None:
        return False
    pred = normalize_answer(predicted)
    gt = normalize_answer(ground_truth)
    # Exact match
    if pred == gt:
        return True
    # Binary strings: EXACT only (rescore Apr 7 fix)
    if len(gt) > 3 and all(c in "01" for c in gt):
        return False
    # Float tolerance for numeric answers
    try:
        pf, gf = float(pred), float(gt)
        if abs(pf - gf) / max(abs(gf), 1e-10) < 0.01:
            return True
    except (ValueError, TypeError):
        pass
    return False

def binary_reward_fn(completions, answer, **kwargs):
    """1.0 if correct answer, 0.0 otherwise."""
    rewards = []
    for i, c in enumerate(completions):
        if isinstance(c, list):
            content = c[0]["content"]
        elif isinstance(c, dict):
            content = c.get("content", "")
        else:
            content = str(c)
        predicted = extract_boxed(content)
        gt = answer[i] if isinstance(answer, (list, tuple)) else answer
        rewards.append(1.0 if verify_answer(predicted, gt) else 0.0)
    return rewards

def format_reward_fn(completions, **kwargs):
    """1.0 if has \\boxed{}, 0.0 otherwise."""
    rewards = []
    for c in completions:
        if isinstance(c, list):
            content = c[0]["content"]
        elif isinstance(c, dict):
            content = c.get("content", "")
        else:
            content = str(c)
        rewards.append(1.0 if "\\boxed{" in content else 0.0)
    return rewards

# Self-test
assert verify_answer("42", "42") == True
assert verify_answer("3.14", "3.15") == True   # within 1%
assert verify_answer("11010", "11011") == False  # binary: exact only
assert verify_answer(None, "42") == False
assert extract_boxed("The answer is \\boxed{42}") == "42"
assert extract_boxed("No boxed here") is None
print("Reward functions: OK (binary + format, 5 tests passed)")

In [ ]:
#@title 5. Carregar modelo (4-bit NF4) + v30 LoRA
print("=== Loading base model (4-bit NF4) ===")
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map={"": 0},
    trust_remote_code=True,
    quantization_config=bnb_config,
)

# CRITICAL FIX 1: Disable Mamba CUDA kernels (incompatible with 4-bit quantization)
import sys as _sys
_mamba_patched = False
for _mod_name, _mod in list(_sys.modules.items()):
    if "modeling_nemotron_h" in _mod_name and hasattr(_mod, "is_fast_path_available"):
        _mod.is_fast_path_available = False
        _mamba_patched = True
        print(f"  Mamba CUDA kernels DISABLED: {_mod_name}")
if not _mamba_patched:
    print("  WARNING: Could not find modeling_nemotron_h to patch!")

# CRITICAL FIX 15: Patch MoE index_add_ dtype mismatch
# Bug: final_hidden_states (BF16/FP32) vs weighted_output (FP32/BF16) under autocast
# Error: "index_add_(): self (BFloat16) and source (Float) must have the same scalar type"
# Fix: monkey-patch the moe() method to cast weighted_output before index_add_
_moe_patched = 0
for _name, _module in model.named_modules():
    cls_name = type(_module).__name__
    if cls_name == "NemotronHMoEBlock" and hasattr(_module, "moe"):
        _orig_moe = _module.moe

        def _make_patched_moe(orig_fn):
            def _patched_moe(hidden_states, topk_indices, topk_weights):
                final_hidden_states = torch.zeros_like(hidden_states, dtype=topk_weights.dtype)
                expert_mask = torch.nn.functional.one_hot(topk_indices, num_classes=len(orig_fn.__self__.experts))
                expert_mask = expert_mask.permute(2, 0, 1)

                for expert_idx in range(len(orig_fn.__self__.experts)):
                    expert = orig_fn.__self__.experts[expert_idx]
                    mask = expert_mask[expert_idx]
                    token_indices, weight_indices = torch.where(mask)

                    if token_indices.numel() > 0:
                        expert_weights = topk_weights[token_indices, weight_indices]
                        expert_input = hidden_states[token_indices]
                        expert_output = expert(expert_input)
                        weighted_output = expert_output * expert_weights.unsqueeze(-1)
                        # FIX: cast to same dtype before index_add_
                        final_hidden_states.index_add_(0, token_indices,
                            weighted_output.to(final_hidden_states.dtype))
                    else:
                        expert_dtype = expert.down_proj.weight.dtype
                        dummy_out = expert(torch.zeros_like(hidden_states[0]).unsqueeze(0).to(expert_dtype))
                        final_hidden_states = final_hidden_states + dummy_out.to(final_hidden_states.dtype)

                return final_hidden_states.type(hidden_states.dtype)
            return _patched_moe

        _module.moe = _make_patched_moe(_orig_moe)
        _moe_patched += 1

print(f"  MoE dtype fix: {_moe_patched} blocks patched")
print(f"Base: {model.num_parameters()/1e9:.1f}B params (4-bit)")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

# v30 SFT adapter - NO merge! (v38 bug: merge+fresh LoRA = mismatch)
print(f"\nLoading v30 adapter: {SFT_REPO}/{SFT_CKPT}")
from peft import PeftModel
model = PeftModel.from_pretrained(
    model, SFT_REPO, subfolder=SFT_CKPT, is_trainable=True)
print("Adapter: trainable=True, NO merge")
model.print_trainable_parameters()

# Freeze MoE routers (prevent catastrophic interference)
frozen = 0
for name, param in model.named_parameters():
    if param.requires_grad and (
        "router" in name.lower() or
        ("expert" in name.lower() and "lora" not in name.lower())
    ):
        param.requires_grad = False
        frozen += 1
if frozen:
    print(f"Frozen: {frozen} MoE router/expert params")

alloc_gb = torch.cuda.memory_allocated() / 1e9
total_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
free_gb = total_gb - alloc_gb
print(f"\nVRAM: {alloc_gb:.1f}/{total_gb:.0f} GB (free: {free_gb:.1f} GB)")

In [ ]:
#@title 6. Patches + teste de geracao
# Fix cache_position=None para NemotronH
_base = model.base_model.model if hasattr(model, "base_model") else model
_orig_prepare = _base.prepare_inputs_for_generation

def _patched_prepare(input_ids, **kwargs):
    if kwargs.get("cache_position") is None:
        kwargs["cache_position"] = torch.arange(
            input_ids.shape[1], device=input_ids.device)
    return _orig_prepare(input_ids, **kwargs)

_base.prepare_inputs_for_generation = _patched_prepare
print("Patch: cache_position OK")

# === TEST GENERATION ===
print("\n=== TEST GENERATION ===")
test_msgs = [{"role": "user",
              "content": "What is 2+2? Put your answer in \\boxed{}."}]
test_text = tokenizer.apply_chat_template(
    test_msgs, tokenize=False, add_generation_prompt=True)
test_ids = tokenizer(test_text, return_tensors="pt").input_ids.to(model.device)

with torch.no_grad():
    out = model.generate(test_ids, max_new_tokens=64,
                         temperature=0.7, do_sample=True)
gen_tokens = len(out[0]) - test_ids.shape[1]
generated = tokenizer.decode(out[0][test_ids.shape[1]:],
                             skip_special_tokens=True)
print(f"Tokens: {gen_tokens}")
print(f"Output: {generated[:300]}")

if gen_tokens <= 1:
    raise RuntimeError(
        "ERRO CRITICO: Modelo gera apenas 1 token!\n"
        "Possivel causa: adapter corrompido ou base model incompativel.\n"
        "Tente: reinstalar peft, ou verificar SFT_REPO/SFT_CKPT"
    )

# === TEST REWARD ===
print("\n=== TEST REWARD ===")
test_comp = [[{"role": "assistant", "content": "Answer is \\boxed{4}"}]]
test_rew = binary_reward_fn(test_comp, answer=["4"])
print(f"binary_reward([boxed 4], gt=4) = {test_rew} (expect [1.0])")
assert test_rew == [1.0], f"Reward test FAILED: {test_rew}"

test_fmt = format_reward_fn(test_comp)
print(f"format_reward([boxed 4]) = {test_fmt} (expect [1.0])")
assert test_fmt == [1.0], f"Format test FAILED: {test_fmt}"

alloc_gb = torch.cuda.memory_allocated() / 1e9
total_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"\nVRAM: {alloc_gb:.1f}/{total_gb:.0f} GB")
print("Todos os testes OK!")

In [ ]:
#@title 7. GRPO Config + Callbacks
from trl import GRPOConfig, GRPOTrainer
from transformers import TrainerCallback
import glob as _glob

os.makedirs("/tmp/kg1_output/v39_grpo", exist_ok=True)

grpo_config = GRPOConfig(
    output_dir="/tmp/kg1_output/v39_grpo",
    # === Optimization (aligned with NVIDIA official grpo_nanov3.yaml) ===
    learning_rate=5e-6,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=2,              # SPEED FIX: 4->2
    max_grad_norm=1.0,              # FIX #14: 0.1->1.0 (NVIDIA oficial usa 1.0)
    weight_decay=0.0,               # FIX #13: 0.1->0.0 (NVIDIA oficial usa 0.0)
    optim="adamw_8bit",
    lr_scheduler_type="cosine",
    warmup_steps=5,
    # === GRPO (NVIDIA official: DAPO style) ===
    num_generations=2,              # SPEED FIX: 4->2 (Mamba torch_forward lento)
    max_completion_length=512,      # SPEED FIX: 1024->512 (4x mais rapido)
    temperature=1.0,                # FIX #12: 0.7->1.0 (NVIDIA oficial usa 1.0)
    beta=0.0,
    loss_type="dapo",               # FIX #11: grpo->dapo (clip_higher + token-level loss)
    # === Schedule ===
    num_train_epochs=1,
    max_steps=GRPO_STEPS,
    logging_steps=1,
    save_strategy="steps",
    save_steps=30,                              # SPEED FIX: alinhado com SUBMIT_STEPS
    save_total_limit=6,
    # === Memory ===
    bf16=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    report_to="none",
    dataloader_num_workers=0,
)

class AutoSubmitCallback(TrainerCallback):
    """Upload to HF + submit to Kaggle on each save."""

    def __init__(self, hf_repo, submit_steps, competition):
        self.hf_repo = hf_repo
        self.submit_steps = set(submit_steps)
        self.competition = competition
        self.submitted = set()
        self.hf_api = HfApi()
        try:
            self.hf_api.create_repo(hf_repo, private=True, exist_ok=True)
        except Exception:
            pass

    def on_log(self, args, state, control, logs=None, **kwargs):
        if not logs or state.global_step % 5 != 0:
            return
        loss = logs.get("loss", "?")
        rew = logs.get("reward", logs.get("mean_reward", "?"))
        cl = logs.get("mean_completion_length",
                      logs.get("completion_length", "?"))
        print(f"  [Step {state.global_step}] loss={loss} "
              f"reward={rew} len={cl}")

    def on_save(self, args, state, control, **kwargs):
        step = state.global_step
        ckpts = sorted(_glob.glob(f"{args.output_dir}/checkpoint-*"))
        if not ckpts:
            return
        ckpt_dir = ckpts[-1]

        # Get loss
        loss_val = "?"
        if state.log_history:
            for entry in reversed(state.log_history):
                if "loss" in entry:
                    loss_val = f"{entry['loss']:.4f}"
                    break

        # HF upload
        try:
            self.hf_api.upload_folder(
                folder_path=ckpt_dir,
                path_in_repo=f"checkpoint-{step}",
                repo_id=self.hf_repo,
                commit_message=f"Step {step} | Loss {loss_val}",
            )
            print(f"  >> HF upload: step {step} OK")
        except Exception as e:
            print(f"  >> HF upload FAIL: {e}")

        # Kaggle submit
        if step not in self.submit_steps or step in self.submitted:
            return
        try:
            zip_path = f"/tmp/kg1_submit/v39_step{step}.zip"
            os.makedirs(os.path.dirname(zip_path), exist_ok=True)
            files_added = 0
            with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
                for fn in ["adapter_config.json",
                           "adapter_model.safetensors"]:
                    fp = os.path.join(ckpt_dir, fn)
                    if os.path.exists(fp):
                        zf.write(fp, fn)
                        files_added += 1
            if files_added < 2:
                print(f"  >> WARN: only {files_added}/2 files in zip")
                return
            desc = (f"v39-DAPO step{step} loss{loss_val} "
                    f"beta0 lr5e-6 4bit n4 eq-focus")
            r = subprocess.run(
                ["kaggle", "competitions", "submit",
                 "-c", self.competition,
                 "-f", zip_path, "-m", desc],
                capture_output=True, text=True, timeout=300,
            )
            if r.returncode == 0:
                print(f"  >> Kaggle: {desc}")
                self.submitted.add(step)
            else:
                print(f"  >> Kaggle FAIL: {r.stderr[:150]}")
        except Exception as e:
            print(f"  >> Submit err: {e}")

import trl
print(f"TRL: {trl.__version__}")
print(f"Config: {GRPO_STEPS} steps | DAPO | num_gen=2 | max_compl=512 | temp=1.0")
print(f"Submit at: {sorted(SUBMIT_STEPS)}")


In [ ]:
#@title 8. TREINAR GRPO
trainer = GRPOTrainer(
    model=model,
    reward_funcs=[binary_reward_fn, format_reward_fn],
    args=grpo_config,
    train_dataset=grpo_dataset,
    processing_class=tokenizer,
    callbacks=[AutoSubmitCallback(OUTPUT_REPO, SUBMIT_STEPS, COMPETITION)],
)

print("=" * 60)
print(f"  v39 GRPO | {GRPO_STEPS} steps | 4-bit NF4 | beta=0")
print(f"  Dataset: {len(grpo_dataset)} prompts | GPU: {gpu_name}")
print("=" * 60)

alloc_gb = torch.cuda.memory_allocated() / 1e9
total_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"VRAM pre-train: {alloc_gb:.1f}/{total_gb:.0f} GB")
sys.stdout.flush()

t0 = time.time()
elapsed = 0
try:
    trainer.train()
    elapsed = time.time() - t0
    print(f"\nGRPO COMPLETE: {elapsed/3600:.2f}h")

    # Save final adapter
    print("Saving final adapter...")
    model.save_pretrained("/tmp/kg1_output/v39_grpo/final")
    tokenizer.save_pretrained("/tmp/kg1_output/v39_grpo/final")
    try:
        api.upload_folder(
            folder_path="/tmp/kg1_output/v39_grpo/final",
            repo_id=OUTPUT_REPO,
            path_in_repo="final",
            commit_message=f"FINAL: {GRPO_STEPS} steps, {elapsed/3600:.1f}h",
        )
        print(f"Upload OK: https://huggingface.co/{OUTPUT_REPO}")
    except Exception as e:
        print(f"Upload FAIL (adapter salvo local): {e}")

except Exception as e:
    elapsed = time.time() - t0
    print(f"\nERRO after {elapsed:.0f}s: {e}")
    import traceback
    traceback.print_exc()
    alloc_gb = torch.cuda.memory_allocated() / 1e9
    total_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"VRAM: {alloc_gb:.1f}/{total_gb:.0f} GB")

print(f"\n{'='*60}")
status = "COMPLETE" if elapsed > 60 else "FAILED"
print(f"  {status} | {elapsed/3600:.2f}h")
print(f"{'='*60}")

In [ ]:
#@title 9. (Opcional) Submit manual + check status
# === Submeter checkpoint manual ===
# Descomente e ajuste STEP:
#
# STEP = 100
# ckpt = f"/tmp/kg1_output/v39_grpo/checkpoint-{STEP}"
# zp = f"/tmp/kg1_submit/manual_step{STEP}.zip"
# os.makedirs(os.path.dirname(zp), exist_ok=True)
# with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as zf:
#     for fn in ["adapter_config.json", "adapter_model.safetensors"]:
#         fp = os.path.join(ckpt, fn)
#         if os.path.exists(fp): zf.write(fp, fn)
# !kaggle competitions submit -c {COMPETITION} -f {zp} -m "v39 manual step{STEP}"

# === Verificar status dos submits ===
# !kaggle competitions submissions -c {COMPETITION} | head -20

print("Descomente para submeter manualmente ou verificar status.")